# Инференс классификатора позиций чеков

Вход — parquet с колонкой `name`; остальные колонки пробрасываются в выход. Если во входе есть `L1..L3`, считаются метрики как при обучении. Запуск: **Run All**.

In [ ]:
import sys

sys.path.insert(0, "..")

from IPython.display import display

from kkt_clf import InferenceConfig, run_inference, setup_logging

setup_logging()

## Конфиг

In [ ]:
CONFIG = {
    "artifact_dir": "../outputs/train_ods/artifact",
    "input_path": "../fixtures/infer_unknown_labels.parquet",
    "output_path": "../outputs/inference_ods/predictions.parquet",
    "report_dir": "../outputs/inference_ods/reports",
    "batch_size": 50_000,
    "tau": None,  # None — τ из артефакта
    "reject_on_mismatch_l2": None,  # None — как в артефакте
    "reject_on_mismatch_l1": None,
    "top_k": 3,
    "device": "auto",
}

icfg = InferenceConfig.from_dict(CONFIG)

## Предсказание

In [ ]:
res = run_inference(icfg)
print("Правило отказа:", res.rule)
print("Выход:", res.output_path, res.predictions.shape)
print(res.timings)

In [ ]:
pred = res.predictions
display(pred.head(20))
display(pred["reject_reason"].value_counts(dropna=False))
display(pred[["mismatch_l2", "mismatch_l1"]].mean())

## Метрики (если во входе есть L1..L3)

In [ ]:
ev = res.evaluation
print("Метки во входе:", res.has_labels)
if ev is not None:
    display(ev["summary"])
    print(f"Покрытие: {ev['coverage']:.4f}; пустых признаков: {ev['n_empty_features']}")
    print("Метки вне маппинга модели:", ev["n_unknown_labels"])
    display(ev["reports"]["no_reject__L3__head"].tail(5))
    display(ev["mismatch_rates"])
    display(ev["top_errors"].head(10))